# Interazione con HDFS tramite Python
Questo notebook mostra come interagire con un cluster Apache HDFS tramite l'interfaccia WebHDFS utilizzando la libreria  in Python.

### Operazioni eseguite:
1. Connessione a HDFS NameNode
2. Caricamento del dataset locale su HDFS
3. Verifica della presenza del file e lettura dei metadati
4. Download del dataset da HDFS salvandolo con un nome differente
5. Verifica della dimensione predefinita di un blocco HDFS
6. Generazione di un file più grande del blocco e analisi della suddivisione in più blocchi
7. Ispezione dei DataNode che ospitano i blocchi e test di tolleranza ai guasti (Fault Tolerance)
8. Verifica della ri-replicazione automatica dei blocchi sui nodi superstiti

In [ ]:
import os
import json
import requests
from urllib.parse import urlparse, urlunparse
from hdfs import InsecureClient

# Client HDFS con risoluzione automatica degli indirizzi IP dei DataNode in ambienti Docker
class DockerHdfsClient(InsecureClient):
    def _request(self, method, url, **kwargs):
        try:
            r = requests.get('http://namenode:9870/jmx?qry=Hadoop:service=NameNode,name=NameNodeInfo', timeout=2)
            nodes = json.loads(r.json()['beans'][0]['LiveNodes'])
            id_to_ip = {k.split(':')[0]: v['infoAddr'].split(':')[0] for k, v in nodes.items()}
        except Exception:
            id_to_ip = {}

        parsed = urlparse(url)
        if parsed.hostname in id_to_ip:
            new_netloc = parsed.netloc.replace(parsed.hostname, id_to_ip[parsed.hostname])
            url = urlunparse(parsed._replace(netloc=new_netloc))
        return super()._request(method, url, **kwargs)

# Connessione al NameNode (WebHDFS porta 9870)
client = DockerHdfsClient('http://namenode:9870', user='hadoop')
print('Root directory status:', client.status('/'))

### 1. File locale del Dataset
Il dataset è montato nel volume locale in `/app/datasets`.

In [ ]:
local_dataset_path = '/app/datasets/Solar_Energy_Production.csv'
hdfs_dir = '/datasets'
hdfs_file_path = f'{hdfs_dir}/Solar_Energy_Production.csv'

print(f'File locale: {local_dataset_path}')
print(f'Dimensione locale: {os.path.getsize(local_dataset_path):,} bytes')

### 2. Caricamento del file su HDFS

In [ ]:
# Creazione directory se non presente
client.makedirs(hdfs_dir)

# Caricamento file
print(f'Caricamento di {local_dataset_path} su {hdfs_file_path}...')
client.upload(hdfs_file_path, local_dataset_path, overwrite=True)
print('Upload completato con successo!')

### 3. Verifica presenza del file su HDFS e ispezione metadati

In [ ]:
# Elenco file nella directory HDFS
hdfs_files = client.list(hdfs_dir)
print(f'File presenti in {hdfs_dir}:', hdfs_files)
assert 'Solar_Energy_Production.csv' in hdfs_files, 'File non trovato su HDFS!'

# Ispezione metadati HDFS
metadata = client.status(hdfs_file_path)
print('\nMetadati del file su HDFS:')
for key, val in metadata.items():
    print(f'  {key}: {val}')

### 4. Download da HDFS con un nome differente

In [ ]:
download_file_path = '/app/notebooks/Solar_Energy_Production_renamed.csv'

# Rimuoviamo il file se precedentemente scaricato
if os.path.exists(download_file_path):
    os.remove(download_file_path)

print(f'Download in corso da HDFS ({hdfs_file_path}) a {download_file_path}...')
client.download(hdfs_file_path, download_file_path)
print('Download completato!')

# Verifica file scaricato
print(f'Dimensione file scaricato: {os.path.getsize(download_file_path):,} bytes')
assert os.path.getsize(download_file_path) == os.path.getsize(local_dataset_path), 'Dimensione diversa dal dataset originale!'

# Pulizia file temporaneo scaricato
os.remove(download_file_path)

### 5. Verifica della dimensione di un blocco HDFS
In HDFS la dimensione di blocco predefinita (configurazione `dfs.blocksize`) è tipicamente **128 MB** (ovvero 134.217.728 byte).
Possiamo verificarla direttamente dai metadati (`blockSize`) di qualsiasi file presente su HDFS.

In [ ]:
# Lettura del blockSize dai metadati del dataset caricato
file_meta = client.status(hdfs_file_path)
block_size_bytes = file_meta['blockSize']
block_size_mb = block_size_bytes / (1024 * 1024)

print(f'Dimensione del blocco HDFS (blockSize): {block_size_bytes:,} bytes ({block_size_mb:.1f} MB)')
print(f'Dimensione totale del dataset: {file_meta["length"]:,} bytes ({file_meta["length"] / (1024 * 1024):.2f} MB)')
if file_meta['length'] < block_size_bytes:
    print('-> Il dataset è più piccolo della dimensione del blocco, quindi risiede in 1 singolo blocco.')

### 6. Generazione di un file più grande del blocco e verifica della suddivisione in più blocchi

Mostriamo la suddivisione dei blocchi e i DataNode che li memorizzano in due scenari:
1. **Scenario su blocco standard (128 MB)**: generiamo un file da **140 MB** diviso in 2 blocchi.
2. **Scenario con blocco personalizzato (4 MB)**: generiamo un file da **10 MB** diviso in 3 blocchi.

In [ ]:
# Funzione di utilità per interrogare WebHDFS per le posizioni e dimensioni dei singoli blocchi
def get_block_locations(hdfs_path):
    url = f'http://namenode:9870/webhdfs/v1{hdfs_path}?op=GETFILEBLOCKLOCATIONS&user.name=hadoop'
    resp = requests.get(url)
    resp.raise_for_status()
    return resp.json()['BlockLocations']['BlockLocation']

# ---------------------------------------------------------------------------
# Test 1: File da 140 MB (> blocco standard da 128 MB)
# ---------------------------------------------------------------------------
test_140mb_path = '/large_file_140mb.bin'
print('Generazione e caricamento in streaming su HDFS di un file da 140 MB...')

chunk_1mb = b'0' * (1024 * 1024)
def generate_140mb():
    for _ in range(140):
        yield chunk_1mb

# Scrittura diretta in streaming senza occupare spazio su disco del container
client.write(test_140mb_path, generate_140mb(), overwrite=True)
print('Upload completato!')

# Ispezione dei blocchi generati dal NameNode
blocks_140mb = get_block_locations(test_140mb_path)
print(f'\nRisultato per {test_140mb_path} (totale 140 MB):')
print(f'Numero totale di blocchi HDFS: {len(blocks_140mb)}')
for i, blk in enumerate(blocks_140mb, start=1):
    length_mb = blk['length'] / (1024 * 1024)
    offset_mb = blk['offset'] / (1024 * 1024)
    print(f'  - Blocco #{i}: {blk["length"]:,} bytes ({length_mb:.2f} MB) | Offset: {blk["offset"]:,} bytes ({offset_mb:.2f} MB) | DataNodes replica: {blk["hosts"]}')

# Pulizia file temporaneo da 140 MB
client.delete(test_140mb_path)
print('\nFile temporaneo 140 MB rimosso da HDFS.')

In [ ]:
# ---------------------------------------------------------------------------
# Test 2: File da 10 MB con blocksize personalizzato a 4 MB
# ---------------------------------------------------------------------------
test_custom_block_path = '/file_custom_block_10mb.bin'
custom_block_size = 4 * 1024 * 1024  # 4 MB
total_size = 10 * 1024 * 1024        # 10 MB

print(f'Caricamento file da {total_size / (1024 * 1024):.0f} MB con blockSize={custom_block_size / (1024 * 1024):.0f} MB...')

def generate_10mb():
    for _ in range(10):
        yield chunk_1mb

client.write(test_custom_block_path, generate_10mb(), overwrite=True, blocksize=custom_block_size)
print('Upload completato!')

blocks_custom = get_block_locations(test_custom_block_path)
print(f'\nRisultato per {test_custom_block_path}:')
print(f'Numero totale di blocchi HDFS: {len(blocks_custom)}')
for i, blk in enumerate(blocks_custom, start=1):
    length_mb = blk['length'] / (1024 * 1024)
    offset_mb = blk['offset'] / (1024 * 1024)
    print(f'  - Blocco #{i}: {blk["length"]:,} bytes ({length_mb:.2f} MB) | Offset: {blk["offset"]:,} bytes ({offset_mb:.2f} MB) | DataNodes replica: {blk["hosts"]}')

# Pulizia file temporaneo
client.delete(test_custom_block_path)
print('\nFile temporaneo 10 MB rimosso da HDFS.')

### 7. Su quali DataNode sono salvati i blocchi e Tolleranza ai Guasti (Fault Tolerance)

Grazie al fattore di replica (`dfs.replication=2`), ogni blocco viene memorizzato su **2 DataNode distinti**.
Anche se un DataNode va offline o subisce un crash, il cluster garantisce la continuità operativa sia in **lettura** che in **scrittura**.

In [ ]:
# 1. Mappatura completa: stato dei DataNode e allocazione dei blocchi del dataset
jmx_resp = requests.get("http://namenode:9870/jmx?qry=Hadoop:service=NameNode,name=NameNodeInfo")
nn_info = jmx_resp.json()["beans"][0]
live_nodes = json.loads(nn_info.get("LiveNodes", "{}"))
dead_nodes = json.loads(nn_info.get("DeadNodes", "{}"))

num_live = len(live_nodes)
num_dead = len(dead_nodes)

print("Stato del cluster HDFS:")
print(f"  - DataNode attivi (Live): {num_live}")
print(f"  - DataNode offline (Dead): {num_dead}")
print("")
print("DataNode ATTIVI (Live):")
for node_id, details in live_nodes.items():
    print(f"  - ID: {node_id} | IP: {details.get('infoAddr')} | Spazio usato: {details.get('usedSpace'):,} bytes")

if dead_nodes:
    print("")
    print("DataNode OFFLINE (Dead):")
    for node_id, details in dead_nodes.items():
        print(f"  - ID: {node_id} | Ultimo contatto: {details.get('lastContact')}s fa | Decommissionato: {details.get('decommissioned')}")

# Ispezione della posizione delle repliche per il dataset
dataset_blocks = get_block_locations("/datasets/Solar_Energy_Production.csv")
print("")
print("Allocazione repliche per /datasets/Solar_Energy_Production.csv:")
for i, blk in enumerate(dataset_blocks, start=1):
    print(f"  - Blocco #{i} ({blk['length']:,} bytes) salvato sui DataNode: {blk['hosts']} (IP: {blk['names']})")


#### Simulazione e Verifica della Fault Tolerance
Puoi arrestare un DataNode dall'host (es: `docker stop minimal-hdfs-deployment-datanode-1`).
Il blocco sottostante dimostra che il cluster continua a rispondere regolarmente:

In [ ]:
# Test 1: Lettura del dataset (anche con un nodo offline, legge dalla replica superstite)
print('1. Test di lettura del dataset con replica attiva...')
with client.read('/datasets/Solar_Energy_Production.csv', length=120) as reader:
    sample_data = reader.read()
    print('   -> Lettura completata con successo!')
    print('   -> Anteprima dati letti:\n', sample_data.decode('utf-8', errors='ignore').strip())

# Test 2: Scrittura di un nuovo file (il NameNode alloca sui DataNode ancora disponibili)
fault_tolerance_file = '/test_fault_tolerance.txt'
test_payload = b'HDFS continua a operare regolarmente anche in presenza di nodi offline!'

print(f'\n2. Test di scrittura nuovo file ({fault_tolerance_file})...')
client.write(fault_tolerance_file, test_payload, overwrite=True)
print('   -> File scritto con successo su HDFS!')

# Test 3: Rilettura e verifica di integrità del nuovo file
with client.read(fault_tolerance_file) as reader:
    verified_payload = reader.read()
    print(f'   -> File riletto e verificato: {verified_payload.decode("utf-8")}')

# Verifica nodi che ospitano le repliche del nuovo file
new_blk_info = get_block_locations(fault_tolerance_file)
print(f'   -> Nodi DataNode che ospitano il nuovo file: {new_blk_info[0]["hosts"]}')

# Pulizia file di test
client.delete(fault_tolerance_file)
print('\n3. File di test rimosso. Cluster HDFS pienamente tollerante ai guasti!')

### 8. Verifica della ri-replicazione automatica dei blocchi sui nodi superstiti

Il cluster è configurato con un intervallo rapido di heartbeat () per rilevare i nodi offline.
Quando un DataNode viene dichiarato **Dead**, il NameNode individua i blocchi sotto-replicati e ordina ai DataNode superstiti di ri-replicare il blocco per ristabilire il fattore di replica ().

La cella sottostante interroga periodicamente l'allocazione dei blocchi del dataset per mostrare in tempo reale come le repliche vengano ridistribuite esclusivamente sui nodi rimasti attivi.

In [ ]:
import time

target_file = "/datasets/Solar_Energy_Production.csv"
print(f"Controllo allocazione blocchi per {target_file}...")

# 1. Interrogazione blocchi e DataNode che attualmente li memorizzano
blocks_info = get_block_locations(target_file)
for i, blk in enumerate(blocks_info, start=1):
    print(f"Blocco #{i} ({blk['length']:,} bytes) salvato sui DataNode:")
    print(f"  -> Hostname nodi: {blk['hosts']}")
    print(f"  -> Indirizzi IP:  {blk['names']}")

# 2. Verifica stato del cluster NameNode (Live vs Dead)
jmx = requests.get("http://namenode:9870/jmx?qry=Hadoop:service=NameNode,name=NameNodeInfo").json()["beans"][0]
live = json.loads(jmx.get("LiveNodes", "{}"))
dead = json.loads(jmx.get("DeadNodes", "{}"))
print("")
print(f"Stato del cluster HDFS: {len(live)} Live / {len(dead)} Dead")
if dead:
    print(f"DataNode dichiarati Dead dopo il timeout (10s): {list(dead.keys())}")
    print(f"Le repliche sono state redistribuite sui nodi attivi: {list(live.keys())}")
